CELL 1: INSTALLATION AND SETUP

In [1]:
print("Installing required packages...")
!pip install -q google-generativeai tensorflow scikit-learn xgboost lightgbm plotly kaleido

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb

# Generative AI
import google.generativeai as genai
from google.colab import userdata, files

# Utilities
import os
import json
import shutil
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("\n✅ All libraries installed successfully!")
print("\n" + "="*70)
print("INSURANCE COST PREDICTION SYSTEM")
print("Made by: Ujan Pradhan & Rishav Prakash")
print("="*70)

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.3 MB/s eta 0:00:00

✅ All libraries installed successfully!

INSURANCE COST PREDICTION SYSTEM
Made by: Ujan Pradhan & Rishav Prakash


CELL 2: DATA LOADING

In [2]:
print("\n📁 Please upload insurance.csv file:")
uploaded = files.upload()

df = pd.read_csv('insurance.csv')
print(f"\n✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())


📁 Please upload insurance.csv file:


Saving insurance.csv to insurance.csv

✅ Dataset loaded: 1338 rows, 7 columns

First 5 rows:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None

Missing Values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Statistical Summary:


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


CELL 3: FEATURE ENGINEERING

In [3]:
def create_features(df):
    """Advanced feature engineering for insurance data"""
    df_new = df.copy()

    # Age groups
    df_new['age_group'] = pd.cut(df_new['age'],
                                  bins=[0, 25, 35, 45, 55, 100],
                                  labels=['18-25', '26-35', '36-45', '46-55', '56+'])

    # BMI categories
    df_new['bmi_category'] = pd.cut(df_new['bmi'],
                                     bins=[0, 18.5, 25, 30, 100],
                                     labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

    # Binary risk factors
    df_new['is_obese'] = (df_new['bmi'] >= 30).astype(int)
    df_new['is_smoker'] = (df_new['smoker'] == 'yes').astype(int)
    df_new['has_children'] = (df_new['children'] > 0).astype(int)
    df_new['is_elderly'] = (df_new['age'] >= 55).astype(int)

    # Interaction features
    df_new['smoking_bmi'] = df_new['is_smoker'] * df_new['bmi']
    df_new['age_bmi'] = df_new['age'] * df_new['bmi']
    df_new['smoking_age'] = df_new['is_smoker'] * df_new['age']

    # Polynomial features
    df_new['age_squared'] = df_new['age'] ** 2
    df_new['bmi_squared'] = df_new['bmi'] ** 2

    # Risk score
    df_new['risk_score'] = (df_new['is_smoker'] * 3 +
                            df_new['is_obese'] * 2 +
                            df_new['is_elderly'] * 1)

    # Encode categorical
    df_new['sex_encoded'] = df_new['sex'].map({'male': 1, 'female': 0})
    df_new['smoker_encoded'] = df_new['is_smoker']

    # Region encoding
    region_encoder = LabelEncoder()
    df_new['region_encoded'] = region_encoder.fit_transform(df_new['region'])

    return df_new

df_processed = create_features(df)
print("\n✅ Feature engineering completed!")
print(f"New dataset shape: {df_processed.shape}")
print(f"New features created: {df_processed.shape[1] - df.shape[1]} additional features")



✅ Feature engineering completed!
New dataset shape: (1338, 22)
New features created: 15 additional features


CELL 4: DATA PREPARATION

In [4]:
feature_columns = [
    'age', 'bmi', 'children', 'sex_encoded', 'region_encoded', 'smoker_encoded',
    'is_obese', 'is_smoker', 'has_children', 'is_elderly',
    'smoking_bmi', 'age_bmi', 'smoking_age',
    'age_squared', 'bmi_squared', 'risk_score'
]

X = df_processed[feature_columns]
y = df_processed['charges']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Data split complete:")
print(f"Training: {X_train_scaled.shape[0]} samples")
print(f"Validation: {X_val_scaled.shape[0]} samples")
print(f"Test: {X_test_scaled.shape[0]} samples")
print(f"Features: {X_train_scaled.shape[1]}")


✅ Data split complete:
Training: 856 samples
Validation: 214 samples
Test: 268 samples
Features: 16


CELL 5: DEEP LEARNING MODEL

In [5]:
def build_deep_model(input_dim):
    """Build deep neural network"""
    model = Sequential([
        Dense(512, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.3),

        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(64, activation='relu'),
        Dropout(0.1),

        Dense(32, activation='relu'),
        Dense(1, activation='linear')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )

    return model

print("\n🚀 Training Deep Learning Model...")
dl_model = build_deep_model(X_train_scaled.shape[1])

print("\nModel Architecture:")
dl_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
]

history = dl_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Deep Learning Model trained!")



🚀 Training Deep Learning Model...

Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │         8,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,881 (730.00 KB)

 Trainable params: 185,089 (723.00 KB)

 Non-trainable params: 1,792 (7.00 KB)

Epoch 1/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - loss: 301620224.0000 - mae: 12867.2627 - val_loss: 354868992.0000 - val_mae: 13983.1475 - learning_rate: 0.0010
Epoch 2/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 300690848.0000 - mae: 12851.9326 - val_loss: 353199296.0000 - val_mae: 13956.8887 - learning_rate: 0.0010
Epoch 3/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 297416512.0000 - mae: 12797.4619 - val_loss: 347535904.0000 - val_mae: 13848.3496 - learning_rate: 0.0010
Epoch 4/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 288770656.0000 - mae: 12647.2910 - val_loss: 333922816.0000 - val_mae: 13566.4932 - learning_rate: 0.0010
Epoch 5/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 271433664.0000 - mae: 12332.8711 - val_loss: 307684736.0000 - val_mae: 13014.4092 - learning_rate: 0.0010
Epoch 6/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 243056784.0000 - mae: 11785.7588 - val_loss: 264609664.0000 - val_mae: 12095.3916 - learning_rate: 0.0010
Epoc

CELL 6: TRADITIONAL ML MODELS

In [6]:
print("\n🚀 Training Traditional ML Models...")

# Random Forest
print("\n1. Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
print("✅ Random Forest trained")

# XGBoost
print("\n2. Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
print("✅ XGBoost trained")

# LightGBM
print("\n3. Training LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_scaled, y_train)
print("✅ LightGBM trained")

print("\n✅ All Traditional ML Models trained!")



🚀 Training Traditional ML Models...

1. Training Random Forest...
✅ Random Forest trained

2. Training XGBoost...
✅ XGBoost trained

3. Training LightGBM...
✅ LightGBM trained

✅ All Traditional ML Models trained!


CELL 7: MODEL EVALUATION

In [7]:
print("\n📊 Evaluating Models...")

# Predictions
dl_pred = dl_model.predict(X_test_scaled, verbose=0).flatten()
rf_pred = rf_model.predict(X_test_scaled)
xgb_pred = xgb_model.predict(X_test_scaled)
lgb_pred = lgb_model.predict(X_test_scaled)

# Ensemble
ensemble_pred = (dl_pred + rf_pred + xgb_pred + lgb_pred) / 4

# Evaluate
models = {
    'Deep Learning': dl_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'Ensemble': ensemble_pred
}

results = {}
for name, pred in models.items():
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R²': r2_score(y_test, pred)
    }

results_df = pd.DataFrame(results).T
print("\n📊 MODEL PERFORMANCE COMPARISON:")
print("="*70)
display(results_df.round(4))
print("="*70)



📊 Evaluating Models...

📊 MODEL PERFORMANCE COMPARISON:


,RMSE,MAE,R²
Deep Learning,4521.9976,2718.8080,0.8683
Random Forest,4553.3685,2380.8865,0.8665
XGBoost,4770.8125,2541.0081,0.8534
LightGBM,4573.4105,2552.3919,0.8653
Ensemble,4411.5992,2298.0852,0.8746


CELL 8: VISUALIZATIONS

In [9]:
print("\n📊 Creating visualizations...")

# Plot 1: Model Performance
fig1 = px.bar(results_df.reset_index(), x='index', y='R²',
              title='Model Performance Comparison (R² Score)',
              color='R²', text='R²')
fig1.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig1.show()

# Plot 2: Predictions vs Actual
fig2 = make_subplots(rows=2, cols=2,
                     subplot_titles=['Deep Learning', 'Random Forest', 'XGBoost', 'LightGBM'])

for idx, (pred, name) in enumerate([(dl_pred, 'DL'), (rf_pred, 'RF'),
                                     (xgb_pred, 'XGB'), (lgb_pred, 'LGB')], 1):
    row = ((idx - 1) // 2) + 1
    col = ((idx - 1) % 2) + 1

    fig2.add_trace(go.Scatter(x=y_test.values, y=pred, mode='markers',
                             name=name, opacity=0.6), row=row, col=col)
    fig2.add_trace(go.Scatter(x=[y_test.min(), y_test.max()],
                             y=[y_test.min(), y_test.max()],
                             mode='lines', line=dict(dash='dash', color='red'),
                             showlegend=False), row=row, col=col)

fig2.update_xaxes(title_text="Actual Costs ($)")
fig2.update_yaxes(title_text="Predicted Costs ($)")
fig2.update_layout(height=900, showlegend=False, title="Predictions vs Actual")
fig2.show()

# Plot 3: Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True).tail(12)

fig3 = go.Figure(go.Bar(x=feature_importance['importance'], y=feature_importance['feature'], orientation='h',
                        marker=dict(color=feature_importance['importance'], colorscale='RdYlGn')))
fig3.update_layout(title='Top 12 Important Features (XGBoost)', height=700,
                   xaxis_title='Importance', yaxis_title='Feature')
fig3.show()

# Plot 4: Training History
epochs_list = list(range(1, len(history.history['loss']) + 1))
fig4 = make_subplots(rows=1, cols=2, subplot_titles=['Loss', 'MAE'])

fig4.add_trace(go.Scatter(x=epochs_list, y=history.history['loss'],
                         name='Train Loss', mode='lines'), row=1, col=1)
fig4.add_trace(go.Scatter(x=epochs_list, y=history.history['val_loss'],
                         name='Val Loss', mode='lines'), row=1, col=1)
fig4.add_trace(go.Scatter(x=epochs_list, y=history.history['mae'],
                         name='Train MAE', mode='lines'), row=1, col=2)
fig4.add_trace(go.Scatter(x=epochs_list, y=history.history['val_mae'],
                         name='Val MAE', mode='lines'), row=1, col=2)

fig4.update_layout(height=500, title='Deep Learning Training Progress')
fig4.show()

print("✅ All visualizations created!")


📊 Creating visualizations...


✅ All visualizations created!


CELL 9: TEST PREDICTIONS

In [10]:
print("\n" + "="*70)
print("TESTING INDIVIDUAL PREDICTIONS")
print("="*70)

def predict_cost(individual_data):
    """Predict insurance cost for an individual"""
    df_ind = pd.DataFrame([individual_data])
    df_ind_processed = create_features(df_ind)
    X_ind = df_ind_processed[feature_columns]
    X_ind_scaled = scaler.transform(X_ind)

    preds = {
        'Deep Learning': float(dl_model.predict(X_ind_scaled, verbose=0)[0][0]),
        'Random Forest': float(rf_model.predict(X_ind_scaled)[0]),
        'XGBoost': float(xgb_model.predict(X_ind_scaled)[0]),
        'LightGBM': float(lgb_model.predict(X_ind_scaled)[0])
    }
    preds['Ensemble'] = (preds['Deep Learning'] + preds['Random Forest'] +
                         preds['XGBoost'] + preds['LightGBM']) / 4

    return preds

# Sample individuals
sample_individuals = [
    {'age': 45, 'sex': 'male', 'bmi': 35.2, 'children': 2, 'smoker': 'yes', 'region': 'southeast'},
    {'age': 28, 'sex': 'female', 'bmi': 22.1, 'children': 0, 'smoker': 'no', 'region': 'northwest'},
    {'age': 55, 'sex': 'male', 'bmi': 28.5, 'children': 3, 'smoker': 'no', 'region': 'northeast'},
]

for i, individual in enumerate(sample_individuals, 1):
    print(f"\n{'='*20} CASE {i} {'='*20}")
    print(f"Profile: {individual['age']}yr {individual['sex']}, BMI {individual['bmi']}, "
          f"{individual['smoker']} smoker, {individual['children']} children")

    preds = predict_cost(individual)

    print(f"\nPredicted Costs:")
    for model_name, cost in preds.items():
        print(f"  {model_name:15}: ${cost:,.2f}")

    print("="*60)


TESTING INDIVIDUAL PREDICTIONS

==================== CASE 1 ====================
Profile: 45yr male, BMI 35.2, yes smoker, 2 children

Predicted Costs:
  Deep Learning  : $42,121.48
  Random Forest  : $42,273.11
  XGBoost        : $44,915.54
  LightGBM       : $47,150.01
  Ensemble       : $44,115.04

==================== CASE 2 ====================
Profile: 28yr female, BMI 22.1, no smoker, 0 children

Predicted Costs:
  Deep Learning  : $203.24
  Random Forest  : $5,416.68
  XGBoost        : $4,572.36
  LightGBM       : $5,191.25
  Ensemble       : $3,845.88

==================== CASE 3 ====================
Profile: 55yr male, BMI 28.5, no smoker, 3 children

Predicted Costs:
  Deep Learning  : $14,419.42
  Random Forest  : $13,336.62
  XGBoost        : $13,083.02
  LightGBM       : $12,626.22
  Ensemble       : $13,366.32


CELL 10: SAVE MODELS AND EXPORT

In [11]:
print("\n" + "="*70)
print("SAVING MODELS AND CREATING EXPORT PACKAGE")
print("="*70)

# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('exports', exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save Models
print("\n💾 Saving models...")
dl_model.save('models/insurance_dl_model.h5')
joblib.dump(rf_model, 'models/insurance_rf_model.pkl')
joblib.dump(xgb_model, 'models/insurance_xgb_model.pkl')
joblib.dump(lgb_model, 'models/insurance_lgb_model.pkl')
joblib.dump(scaler, 'models/insurance_scaler.pkl')

with open('models/feature_columns.txt', 'w') as f:
    f.write('\n'.join(feature_columns))

print("✅ All models saved!")

# Save Predictions
print("\n📊 Saving predictions and metrics...")
predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'dl': dl_pred,
    'rf': rf_pred,
    'xgb': xgb_pred,
    'lgb': lgb_pred,
    'ensemble': ensemble_pred
})
predictions_df.to_csv('exports/predictions.csv', index=False)

results_df.to_csv('exports/model_metrics.csv')

feature_importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
feature_importance_df.to_csv('exports/feature_importance.csv', index=False)

print("✅ Data exported!")

# Save plots
print("\n📊 Saving plots...")
try:
    fig1.write_html('plots/01_model_performance.html')
    fig2.write_html('plots/02_predictions_vs_actual.html')
    fig3.write_html('plots/03_feature_importance.html')
    fig4.write_html('plots/04_training_history.html')
    print("✅ Plots saved as HTML!")
except Exception as e:
    print(f"⚠️ Plot save error: {e}")


SAVING MODELS AND CREATING EXPORT PACKAGE

💾 Saving models...
✅ All models saved!

📊 Saving predictions and metrics...
✅ Data exported!

📊 Saving plots...
✅ Plots saved as HTML!
